# Coding Attention Mechanism

### Simple attention mechanism

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch

In [3]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], ## your
        [0.55, 0.87, 0.66], # journey
        [0.57, 0.85, 0.64], # starts
        [0.22, 0.58, 0.33], # with
        [0.77, 0.25, 0.10], # one
        [0.05, 0.80, 0.55] # steps
    ]
)

In [4]:
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [5]:
input_1 = inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [6]:
torch.dot(input_query, input_1)

tensor(0.9544)

In [7]:
for i in range(len(inputs)):
    res = torch.dot(inputs[i], input_query)
    print(res)

tensor(0.9544)
tensor(1.4950)
tensor(1.4754)
tensor(0.8434)
tensor(0.7070)
tensor(1.0865)


In [8]:
query = inputs[1]

attn_score = torch.empty(inputs.shape[0])

for i, i_x in enumerate(inputs):
    attn_score[i] = torch.dot(i_x, query)

attn_score = torch.softmax(attn_score, dim=0)
attn_score

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [9]:
query = inputs[1]

context_vec = torch.zeros(query.shape)

for i, i_x in enumerate(inputs):
    context_vec += attn_score[i] * i_x
print(context_vec)

tensor([0.4419, 0.6515, 0.5683])


### Simple attention without trainable parameter

In [33]:
attn_scores = inputs @ inputs.T
attn_weight = torch.softmax(attn_scores, dim=1)
context_vec = attn_weight @ inputs
print(context_vec)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### With trainable weight

In [34]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [68]:
x_2 = inputs[1]
in_d = inputs.shape[1]
out_d = 2

In [69]:
torch.manual_seed(123)

w_query = torch.nn.Parameter(torch.rand(in_d, out_d))
w_key = torch.nn.Parameter(torch.rand(in_d, out_d))
w_value = torch.nn.Parameter(torch.rand(in_d, out_d))

In [70]:
query = x_2 @ w_query
query

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [71]:
key = inputs @ w_key
value = inputs @ w_value

In [72]:
key.shape

torch.Size([6, 2])

In [73]:
attn_score = torch.dot(query, key[1])
attn_score

tensor(1.8524, grad_fn=<DotBackward0>)

In [74]:
attn_scores = torch.matmul(query, key.T)
attn_scores

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [75]:
d_k = key.shape[1]


attn_weight_2 = torch.softmax(attn_scores / d_k**0.5, dim=-1)
attn_weight_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [76]:
context_vec_2 = attn_weight_2 @ value
context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

## Implementing a compactable class

In [79]:
class SelfAttentionV1(torch.nn.Module):

    def __init__(self, in_d, out_d):
        super().__init__()
        self.w_query = torch.nn.Parameter(torch.rand(in_d, out_d))
        self.w_key = torch.nn.Parameter(torch.rand(in_d, out_d))
        self.w_value = torch.nn.Parameter(torch.rand(in_d, out_d))


    def forward(self):
        query = inputs @ self.w_query
        key = inputs @ self.w_key
        value = inputs @ self.w_value

        attn_scores = torch.matmul(query, key.T)
        attn_weight = torch.softmax(attn_scores / key.shape[1]**0.5, dim=-1)
        context_vec = attn_weight @ value
        return context_vec

torch.manual_seed(123)
attention = SelfAttentionV1(in_d, out_d)
attention()

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [88]:
class SelfAttentionV2(torch.nn.Module):

    def __init__(self, in_d, out_d, qkv_bias = False):
        super().__init__()
        self.w_query = torch.nn.Linear(in_d, out_d, bias=qkv_bias)
        self.w_key = torch.nn.Linear(in_d, out_d, bias=qkv_bias)
        self.w_value = torch.nn.Linear(in_d, out_d, bias=qkv_bias)


    def forward(self, inputs):
        query = self.w_query(inputs)
        key = self.w_key(inputs)
        value = self.w_value(inputs)

        attn_scores = torch.matmul(query, key.T)
        attn_weight = torch.softmax(attn_scores / key.shape[1]**0.5, dim=-1)
        context_vec = attn_weight @ value
        return context_vec

torch.manual_seed(789)
attentionv2 = SelfAttentionV2(in_d, out_d)
attentionv2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

### applying causal attention mask
- hideing future token

In [89]:
query = attentionv2.w_query(inputs)
key = attentionv2.w_key(inputs)
value = attentionv2.w_value(inputs)

attn_scores = torch.matmul(query, key.T)
attn_weight = torch.softmax(attn_scores / key.shape[1]**0.5, dim=-1)

In [90]:
attn_scores

tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)

In [92]:
attn_scores.shape

torch.Size([6, 6])

In [94]:
contex_len = attn_scores.shape[0]
masked_matrix = torch.tril(torch.ones(contex_len, contex_len))
masked_matrix

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [95]:
simple_masked = attn_scores * masked_matrix
simple_masked

tensor([[0.2899, 0.0000, 0.0000, -0.0000, 0.0000, -0.0000],
        [0.4656, 0.1723, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4594, 0.1703, 0.1731, 0.0000, 0.0000, 0.0000],
        [0.2642, 0.1024, 0.1036, 0.0186, 0.0000, 0.0000],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786, 0.0000],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MulBackward0>)

In [96]:
rows_sum = simple_masked.sum(dim=-1, keepdim=True)
simple_masked_norm = simple_masked / rows_sum
print(simple_masked_norm)

tensor([[1.0000, 0.0000, 0.0000, -0.0000, 0.0000, -0.0000],
        [0.7300, 0.2700, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5723, 0.2121, 0.2156, 0.0000, 0.0000, 0.0000],
        [0.5404, 0.2095, 0.2120, 0.0381, 0.0000, 0.0000],
        [0.4454, 0.1782, 0.1799, 0.0361, 0.1603, 0.0000],
        [0.4523, 0.1686, 0.1713, 0.0263, 0.1712, 0.0103]],
       grad_fn=<DivBackward0>)


## optimize way

In [97]:
mask = torch.triu(torch.ones(contex_len, contex_len), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [100]:
attn_weight = torch.softmax(masked / d_k**0.5, dim=-1)

In [101]:
attn_weight

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

### Causal attention with dropout

In [102]:
torch.manual_seed(123)
layer = torch.nn.Dropout(0.5)

In [103]:
example = torch.ones(6, 6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [ ]:
## Formula -> 1 / (1 - dropout_rate)
layer(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [105]:
layer(attn_weight)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4925, 0.4638, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3941, 0.0000],
        [0.3869, 0.3327, 0.0000, 0.3084, 0.3331, 0.3058]],
       grad_fn=<MulBackward0>)

# Implement masked self attention in compatable class

In [108]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [112]:
class CausalSelfAttention(torch.nn.Module):

    def __init__(self, in_d, out_d, dropout, context_len, qkv_bias = False):
        super().__init__()
        self.w_query = torch.nn.Linear(in_d, out_d, bias=qkv_bias)
        self.w_key = torch.nn.Linear(in_d, out_d, bias=qkv_bias)
        self.w_value = torch.nn.Linear(in_d, out_d, bias=qkv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1))


    def forward(self, inputs):
        b, num_token, d_in = inputs.shape

        query = self.w_query(inputs)
        key = self.w_key(inputs)
        value = self.w_value(inputs)

        attn_scores = query @ key.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_token, :num_token], -torch.inf)
        attn_weight = torch.softmax(attn_scores / key.shape[-1]**0.5, dim=1)
        attn_weight = self.dropout(attn_weight)

        context_vec = attn_weight @ value
        return context_vec

torch.manual_seed(789)
dropout = 0
contex_len = batch.shape[1]
ca = CausalSelfAttention(in_d, out_d, dropout, contex_len)
ca(batch)

tensor([[[-0.0140,  0.0046],
         [-0.0392,  0.0209],
         [-0.0655,  0.0449],
         [-0.0915,  0.0415],
         [-0.0220,  0.2167],
         [-0.2306,  0.0700]],

        [[-0.0140,  0.0046],
         [-0.0392,  0.0209],
         [-0.0655,  0.0449],
         [-0.0915,  0.0415],
         [-0.0220,  0.2167],
         [-0.2306,  0.0700]]], grad_fn=<UnsafeViewBackward0>)